# Colab Notebook: Phase 2 Encoder Baselines

Use this markdown file as the source for a Colab notebook. Run cells in order.

## 1. Clone repository

In [1]:
import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")

CUDA: True
GPU: Tesla T4


In [2]:
from pathlib import Path

WORKSPACE = Path("/content/Authorship-Attribution")
REPO_PARENT = WORKSPACE / "repo"
REPO = REPO_PARENT / "Authorship-Attribution-in-Victorian-Periodicals"
REPO_PARENT.mkdir(parents=True, exist_ok=True)

if not REPO.exists():
    !git clone https://github.com/IamTaoHu/Authorship-Attribution-in-Victorian-Periodicals.git "{REPO}"
%cd "{REPO}"

Cloning into '/content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals'...
remote: Enumerating objects: 255, done.
remote: Counting objects: 100% (255/255), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 255 (delta 93), reused 225 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (255/255), 195.84 KiB | 1.53 MiB/s, done.
Resolving deltas: 100% (93/93), done.
/content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals


## 2. Install requirements

In [3]:
!pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 80.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 17.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## 3. Optional Hugging Face login

In [4]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("Hugging Face login successful.")
else:
    print("HF_TOKEN not found in Colab Secrets.")

Hugging Face login successful.


In [5]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '69fb5a2a33be6258860f7fb0', 'name': 'IMtaohu', 'fullname': 'Im taohu', 'email': 'pawatisaraporn@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1780272000, 'isPro': False, 'avatarUrl': '/avatars/c35373794b0b1da9fede8272ca77f971.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'HF_TOKEN', 'role': 'read', 'createdAt': '2026-05-12T21:25:34.124Z'}}}


## 4. Configure Colab paths

In [7]:
from pathlib import Path

paths_yaml = f"""paths:
  workspace_root: {WORKSPACE}
  repo_root: {REPO}
  artifacts_root: {WORKSPACE / "artifacts"}
  checkpoints_root: {WORKSPACE / "checkpoints"}
  datasets_root: {WORKSPACE / "datasets"}
  exports_root: {WORKSPACE / "exports"}

runtime:
  default_environment: colab
  cloud_training_environment: colab
"""

In [8]:
Path("configs/paths.local.yaml").write_text(paths_yaml, encoding="utf-8")
print(paths_yaml)

paths:
  workspace_root: /content/Authorship-Attribution
  repo_root: /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
  artifacts_root: /content/Authorship-Attribution/artifacts
  checkpoints_root: /content/Authorship-Attribution/checkpoints
  datasets_root: /content/Authorship-Attribution/datasets
  exports_root: /content/Authorship-Attribution/exports

runtime:
  default_environment: colab
  cloud_training_environment: colab



## 5. Run structure check

In [9]:
!python scripts/check_project_structure.py

Current working directory: /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
Repository root: /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
[OK] configs
[MISSING] docs
[OK] notebooks/local
[OK] notebooks/colab
[OK] notebooks/archive
[OK] prompts
[OK] scripts
[OK] src
[OK] src/utils
[OK] src/data
[OK] src/training
[OK] src/evaluation
[OK] src/visualization
[OK] tests
[OK] configs/paths.example.yaml
[OK] configs/datasets.yaml
[OK] configs/models.yaml
[OK] src/utils/paths.py
[OK] Imported src.utils.paths
Resolved paths:
  workspace_root: /content/Authorship-Attribution
  repo_root: /content/Authorship-Attribution/repo/Authorship-Attribution-in-Victorian-Periodicals
  artifacts_root: /content/Authorship-Attribution/artifacts
  checkpoints_root: /content/Authorship-Attribution/checkpoints
  datasets_root: /content/Authorship-Attribution/datasets
  exports_root: /content/Authorship-Attribution/exports
[OK] Created test

## 6. Verify Phase 1 processed PERIAD data or regenerate Phase 1

In [10]:
from pathlib import Path
import yaml

config = yaml.safe_load(Path("configs/paths.local.yaml").read_text())
periad_dir = Path(config["paths"]["datasets_root"]) / "processed" / "periad"
required = ["train.csv", "test.csv", "label_map.json"]
missing = [name for name in required if not (periad_dir / name).exists()]
print("PERIAD directory:", periad_dir)
print("Missing:", missing)

PERIAD directory: /content/Authorship-Attribution/datasets/processed/periad
Missing: ['train.csv', 'test.csv', 'label_map.json']


In [11]:
if missing:
    !python src/data/prepare_phase1_dataset.py --config configs/phase1/dataset_pipeline.yaml
    !python src/data/dataset_statistics.py --config configs/phase1/dataset_pipeline.yaml
    !python src/data/tokenization_analysis.py --config configs/phase1/dataset_pipeline.yaml
    !python src/visualization/plot_phase1_dataset.py --config configs/phase1/dataset_pipeline.yaml
    !python scripts/check_phase1_outputs.py --config configs/phase1/dataset_pipeline.yaml
else:
    print("Phase 1 processed PERIAD data exists.")

README.md: 100% 360/360 [00:00<00:00, 1.76MB/s]
training.csv: 9.25MB [00:00, 11.6MB/s]
testing.csv: 4.02MB [00:00, 11.5MB/s]
Generating train split: 100% 8279/8279 [00:00<00:00, 65493.85 examples/s]
Generating test split: 100% 3549/3549 [00:00<00:00, 71082.43 examples/s]
Wrote /content/Authorship-Attribution/artifacts/phase1/reports/periad_validation_report.json
Wrote /content/Authorship-Attribution/artifacts/phase1/reports/periad_validation_report.md
Wrote /content/Authorship-Attribution/datasets/processed/periad/train.csv
Wrote /content/Authorship-Attribution/datasets/processed/periad/train.jsonl
Wrote /content/Authorship-Attribution/datasets/processed/periad/test.csv
Wrote /content/Authorship-Attribution/datasets/processed/periad/test.jsonl
Wrote /content/Authorship-Attribution/datasets/processed/periad/label_map.json
Wrote /content/Authorship-Attribution/datasets/processed/periad/dataset_card.json
Wrote /content/Authorship-Attribution/datasets/processed/veaa/metadata_summary.json
W

## 7. Train BERT-base

In [12]:
!python src/training/train_encoder_baseline.py --config configs/phase2/bert_base.yaml

config.json: 100% 570/570 [00:00<00:00, 2.25MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 200kB/s]
vocab.txt: 232kB [00:00, 5.95MB/s]
tokenizer.json: 466kB [00:00, 4.68MB/s]
Map: 100% 8279/8279 [00:05<00:00, 1613.90 examples/s]
Map: 100% 3549/3549 [00:02<00:00, 1607.55 examples/s]
Wrote /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/run_config_resolved.yaml
Wrote /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/label_map.json
model.safetensors: 100% 440M/440M [00:02<00:00, 156MB/s]
Loading weights: 100% 199/199 [00:00<00:00, 1444.43it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.p

## 8. Train RoBERTa-base

In [13]:
!python src/training/train_encoder_baseline.py --config configs/phase2/roberta_base.yaml

config.json: 100% 481/481 [00:00<00:00, 2.41MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 159kB/s]
vocab.json: 899kB [00:00, 11.2MB/s]
merges.txt: 456kB [00:00, 5.86MB/s]
tokenizer.json: 1.36MB [00:00, 5.51MB/s]
Map: 100% 8279/8279 [00:05<00:00, 1490.12 examples/s]
Map: 100% 3549/3549 [00:01<00:00, 1883.08 examples/s]
Wrote /content/Authorship-Attribution/artifacts/phase2/runs/roberta_base/run_config_resolved.yaml
Wrote /content/Authorship-Attribution/artifacts/phase2/runs/roberta_base/label_map.json
model.safetensors: 100% 499M/499M [00:03<00:00, 146MB/s]
Loading weights: 100% 197/197 [00:00<00:00, 1090.64it/s, Materializing param=roberta.encoder.layer.11.output.dense.weight]
RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED |

## 9. Train RoBERTa-large

In [14]:
!python src/training/train_encoder_baseline.py --config configs/phase2/roberta_large.yaml

config.json: 100% 482/482 [00:00<00:00, 2.40MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 143kB/s]
vocab.json: 899kB [00:00, 5.97MB/s]
merges.txt: 456kB [00:00, 6.91MB/s]
tokenizer.json: 1.36MB [00:00, 9.14MB/s]
Map: 100% 8279/8279 [00:05<00:00, 1488.85 examples/s]
Map: 100% 3549/3549 [00:01<00:00, 1876.38 examples/s]
Wrote /content/Authorship-Attribution/artifacts/phase2/runs/roberta_large/run_config_resolved.yaml
Wrote /content/Authorship-Attribution/artifacts/phase2/runs/roberta_large/label_map.json
model.safetensors: 100% 1.42G/1.42G [00:12<00:00, 116MB/s] 
Loading weights: 100% 389/389 [00:00<00:00, 751.46it/s, Materializing param=roberta.encoder.layer.23.output.dense.weight]
RobertaForSequenceClassification LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPEC

## 10. Aggregate results

In [15]:
!python src/evaluation/aggregate_phase2_results.py

Wrote /content/Authorship-Attribution/artifacts/phase2/tables/encoder_results.csv
Wrote /content/Authorship-Attribution/artifacts/phase2/tables/encoder_results.md
Wrote /content/Authorship-Attribution/artifacts/phase2/reports/phase2_encoder_baseline_report.md


## 11. Plot results


In [16]:
!python src/visualization/plot_phase2_results.py

Wrote /content/Authorship-Attribution/artifacts/phase2/plots/encoder_accuracy_bar.png
Wrote /content/Authorship-Attribution/artifacts/phase2/plots/encoder_macro_f1_bar.png
Wrote /content/Authorship-Attribution/artifacts/phase2/plots/bert_base_confusion_matrix.png
Wrote /content/Authorship-Attribution/artifacts/phase2/plots/roberta_base_confusion_matrix.png
Wrote /content/Authorship-Attribution/artifacts/phase2/plots/roberta_large_confusion_matrix.png


## 12. Check outputs


In [17]:
!python scripts/check_phase2_outputs.py

[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/predictions.csv
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/eval_metrics.json
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/classification_report.csv
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/confusion_matrix.csv
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/run_config_resolved.yaml
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/trainer_state.json
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/training_log.csv
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/bert_base/predictions.csv contains prob_0 through prob_5
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/roberta_base/predictions.csv
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/roberta_base/eval_metrics.json
[OK] /content/Authorship-Attribution/artifacts/phase2/runs/roberta_base/classific

## 13. Zip artifacts and checkpoints

In [18]:
from pathlib import Path

artifacts_zip = Path("/content/phase2_artifacts.zip")
checkpoints_zip = Path("/content/phase2_checkpoints.zip")
!cd "{WORKSPACE}" && zip -r "{artifacts_zip}" artifacts/phase2
!cd "{WORKSPACE}" && zip -r "{checkpoints_zip}" checkpoints/phase2
print(artifacts_zip)
print(checkpoints_zip)

  adding: artifacts/phase2/ (stored 0%)
  adding: artifacts/phase2/reports/ (stored 0%)
  adding: artifacts/phase2/reports/phase2_encoder_baseline_report.md (deflated 62%)
  adding: artifacts/phase2/runs/ (stored 0%)
  adding: artifacts/phase2/runs/roberta_large/ (stored 0%)
  adding: artifacts/phase2/runs/roberta_large/predictions.csv (deflated 62%)
  adding: artifacts/phase2/runs/roberta_large/trainer_state.json (deflated 75%)
  adding: artifacts/phase2/runs/roberta_large/classification_report.csv (deflated 46%)
  adding: artifacts/phase2/runs/roberta_large/train_metrics.json (deflated 35%)
  adding: artifacts/phase2/runs/roberta_large/training_log.csv (deflated 61%)
  adding: artifacts/phase2/runs/roberta_large/run_config_resolved.yaml (deflated 66%)
  adding: artifacts/phase2/runs/roberta_large/runtime.json (deflated 31%)
  adding: artifacts/phase2/runs/roberta_large/confusion_matrix.csv (deflated 42%)
  adding: artifacts/phase2/runs/roberta_large/label_map.json (deflated 23%)
  ad

## 14. Download or sync

In [19]:
from google.colab import files

files.download("/content/phase2_artifacts.zip")
files.download("/content/phase2_checkpoints.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Optional Google Drive sync:

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p "/content/drive/MyDrive/Authorship-Attribution/phase2"
!cp /content/phase2_artifacts.zip "/content/drive/MyDrive/Authorship-Attribution/phase2/"